# long-route-planner: plan a 30 km route through villages

# Imports

In [ ]:
import json
from dataclasses import asdict, replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from uavloc.data.tiles import ESRI_WORLD_IMAGERY
from uavloc.features.onnx_segmenter import OnnxSegmenter
from uavloc.geo import LatLon
from uavloc.planning.config import MissionInputs, PlannerConfig
from uavloc.planning.localizability import SelfSimilarity
from uavloc.planning.planner import PlanningResult, plan
from uavloc.planning.routes import Route
from uavloc.simulation.sequence import load_flight_set
from uavloc.track import REPO_ROOT

# Params

In [ ]:
# Village chain east of Busk, Lviv oblast: OSM place nodes, shortest village-to-village path (legs <= 4.5 km)
# along the best-scoring 30 km east-west line (most villages within 600 m)
VIA_POINTS = [(49.9823, 24.7705), (49.9603, 24.8147), (49.9739, 24.8507), (49.9693, 24.8614), (49.9629, 24.8713),
              (49.9643, 24.8941), (49.9647, 24.9303), (49.9635, 24.9432), (49.9591, 24.9779), (49.9620, 25.0405),
              (49.9631, 25.0816), (49.9627, 25.1099), (49.9634, 25.1199), (49.9601, 25.1521), (49.9628, 25.1754)]
ROUTE_SET = "sim-flights"        # flight settings (altitude, speed, INS model) the filter was tuned on
TILE_CACHE = "data/derived/tile-cache"
PLANNER_CACHE = "data/derived/planner-cache"
SATELLITE_ONNX = "pocs/lraspp-pi5/models/lraspp-satellite.onnx"   # the route planner app's segmenter
OUT_DIR = "pocs/long-route-planner/runs/plan"
FIGURE_DIR = "pocs/long-route-planner/figures"

# Data

In [ ]:
set_config, _ = load_flight_set(ROUTE_SET)
via = [LatLon(*p) for p in VIA_POINTS]
legs = list(zip(via, via[1:]))
out_dir, figure_dir = REPO_ROOT / OUT_DIR, REPO_ROOT / FIGURE_DIR
out_dir.mkdir(parents=True, exist_ok=True)
len(legs)

# Model

In [ ]:
config = PlannerConfig(ins=set_config.ins)
segmenter = OnnxSegmenter(REPO_ROOT / SATELLITE_ONNX)
localizability = SelfSimilarity(config.matcher, config.match_gsd_m, config.blind_s_max_m, config.blind_c_min)

In [ ]:
def plan_leg(start: LatLon, end: LatLon) -> PlanningResult:
    inputs = MissionInputs(start, end, set_config.altitude_m, set_config.speed_mps)
    return plan(inputs, config, ESRI_WORLD_IMAGERY, segmenter, localizability, REPO_ROOT / TILE_CACHE,
                REPO_ROOT / PLANNER_CACHE)

In [ ]:
def best_route(routes: list[Route]) -> Route:
    """The planner's alternative with the least blind ground; the shorter one on a tie."""
    return min(routes, key=lambda r: (round(r.blind_share, 3), r.length_m))

In [ ]:
def chained(routes: list[Route]) -> list[LatLon]:
    """One waypoint list over consecutive legs; each leg's end is the next leg's start."""
    points = list(routes[0].waypoints)
    for route in routes[1:]:
        points += list(route.waypoints[1:])
    return points

# Run

In [ ]:
results = [plan_leg(start, end) for start, end in tqdm(legs, desc="legs")]

In [ ]:
best = [best_route(r.routes) for r in results]
shortest = [min(r.routes, key=lambda route: route.length_m) for r in results]

In [ ]:
legs_table = pd.DataFrame([{
    "leg": i, "area_km2": np.prod(r.area.size_m) / 1e6, "alternatives": len(r.routes),
    "best_weight": b.weight, "best_km": b.length_m / 1e3, "best_blind_share": b.blind_share,
    "best_longest_blind_m": b.longest_blind_m, "shortest_km": s.length_m / 1e3,
    "shortest_blind_share": s.blind_share, "shortest_longest_blind_m": s.longest_blind_m}
    for i, (r, b, s) in enumerate(zip(results, best, shortest))])
legs_table.round(3)

In [ ]:
def route_totals(routes: list[Route]) -> dict:
    length_m = sum(r.length_m for r in routes)
    return {"length_km": length_m / 1e3,
            "blind_share": sum(r.blind_share * r.length_m for r in routes) / length_m,
            "longest_blind_m": max(r.longest_blind_m for r in routes),
            "blind_limit_m": routes[0].blind_limit_m}

In [ ]:
totals = {"best": route_totals(best), "shortest": route_totals(shortest)}
pd.DataFrame(totals).round(3)

In [ ]:
routes_out = {"via_points": VIA_POINTS, "planner": json.loads(json.dumps(asdict(config), default=str)),
              "totals": totals, "legs": legs_table.to_dict("records"),
              "routes": {"best": [tuple(p) for p in chained(best)], "shortest": [tuple(p) for p in chained(shortest)]}}
(out_dir / "routes.json").write_text(json.dumps(routes_out, indent=1))

In [ ]:
np.savez_compressed(out_dir / "blind_maps.npz",
                    **{f"leg{i}": r.blind_map.pixels for i, r in enumerate(results)},
                    **{f"leg{i}_bounds": np.array([*r.blind_map.top_left, *r.blind_map.bottom_right])
                       for i, r in enumerate(results)})

# Plots

In [ ]:
fig, ax = plt.subplots(figsize=(18, 4.5))
for r in results:
    (north, west), (south, east) = r.blind_map.top_left, r.blind_map.bottom_right
    ax.imshow(r.blind_map.pixels[..., 0], extent=(west, east, south, north), cmap="RdYlGn_r", vmin=0, vmax=1,
              alpha=0.8, interpolation="nearest")
for name, routes, colour in (("shortest", shortest, "tab:blue"), ("best (planner)", best, "black")):
    points = chained(routes)
    ax.plot([p.lon for p in points], [p.lat for p in points], color=colour, lw=1.4, label=name)
ax.scatter([p.lon for p in via], [p.lat for p in via], c="white", edgecolors="black", s=40, zorder=5, label="villages")
ax.set_aspect(1 / np.cos(np.radians(via[0].lat)))
ax.set_title(f"Planner blind map per leg (red = fix unlikely); best {totals['best']['length_km']:.1f} km, "
             f"{totals['best']['blind_share']:.0%} blind; shortest {totals['shortest']['length_km']:.1f} km, "
             f"{totals['shortest']['blind_share']:.0%} blind")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(figure_dir / "plan.png", dpi=130)